# 1- Carga y filtrado inicial

In [2]:
import pandas as pd

# Cargar el dataset principal
file_path = r'E:/Olise TP/CSV/olist_orders_dataset.csv'
orders_df = pd.read_csv(file_path)

# Aislar únicamente los pedidos que llegaron a destino para no falsear métricas
orders_delivered = orders_df[orders_df['order_status'] == 'delivered'].copy()

print(f"Pedidos originales: {len(orders_df)} | Pedidos entregados: {len(orders_delivered)}")

Pedidos originales: 99441 | Pedidos entregados: 96478


# Celda 2: Transformación a formato DateTime

In [3]:
# Lista de columnas que representan fechas y horas
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

# Convertir de texto a objeto datetime de pandas
for col in date_columns:
    orders_delivered[col] = pd.to_datetime(orders_delivered[col])

# Verificar que los tipos de datos ahora digan datetime64[ns]
print(orders_delivered.dtypes)

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


# Celda 3: Sanitización de claves primarias y foráneas

In [4]:
# Eliminar posibles espacios en blanco invisibles y asegurar formato string
orders_delivered['order_id'] = orders_delivered['order_id'].astype(str).str.strip()
orders_delivered['customer_id'] = orders_delivered['customer_id'].astype(str).str.strip()

# Comprobar si quedaron valores nulos después del filtro
print("\nValores nulos por columna:")
print(orders_delivered.isnull().sum())


Valores nulos por columna:
order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64


# Celda 4: Carga de ítems, productos y diccionario de traducción

In [5]:
# Cargar las tablas relacionadas al producto y al detalle de compra

items_df = pd.read_csv(r'E:\Olise TP\CSV\olist_order_items_dataset.csv')
products_df = pd.read_csv(r'E:\Olise TP\CSV\olist_products_dataset.csv')
translations_df = pd.read_csv(r'E:\Olise TP\CSV\product_category_name_translation.csv')

# Sanitizar las claves foráneas en ítems y productos por seguridad
items_df['order_id'] = items_df['order_id'].astype(str).str.strip()
items_df['product_id'] = items_df['product_id'].astype(str).str.strip()
products_df['product_id'] = products_df['product_id'].astype(str).str.strip()

# Celda 5: Traducción y limpieza del catálogo de productos

In [6]:
# Unir los productos con su traducción al inglés
products_translated = pd.merge(products_df, translations_df, on='product_category_name', how='left')

# Imputar nulos: Algunos productos no tienen categoría asignada. 
# En lugar de borrarlos (y perder la venta), los categorizamos como 'unknown' (desconocido)
products_translated['product_category_name_english'] = products_translated['product_category_name_english'].fillna('unknown')

# Seleccionar solo las columnas útiles para el modelo de datos final
products_clean = products_translated[[
    'product_id', 
    'product_category_name_english', 
    'product_weight_g', 
    'product_length_cm', 
    'product_height_cm', 
    'product_width_cm'
]]

# Celda 6: Consolidación (El gran Merge)

In [7]:
# 1. Unir Pedidos Entregados con sus respectivos Ítems (inner join asegura que solo queden ventas concretadas)
orders_items = pd.merge(orders_delivered, items_df, on='order_id', how='inner')

# 2. Unir el resultado con los detalles del Producto (left join para no perder ventas si un producto fue borrado del catálogo)
master_df = pd.merge(orders_items, products_clean, on='product_id', how='left')

print(f"Filas en el dataset consolidado: {len(master_df)}")
print("\nPrimeros registros de precios, fletes y categorías:")
display(master_df[['order_id', 'product_category_name_english', 'price', 'freight_value']].head())

Filas en el dataset consolidado: 110197

Primeros registros de precios, fletes y categorías:


,order_id,product_category_name_english,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,housewares,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,perfumery,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,auto,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,pet_shop,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,stationery,19.90,8.72


# Celda 7: Tiempos logísticos y demoras

In [8]:
# Calcular el tiempo real de entrega en días (desde que compra hasta que recibe)
# Usamos .dt.days para extraer solo el número entero de días
master_df['delivery_time_days'] = (master_df['order_delivered_customer_date'] - master_df['order_purchase_timestamp']).dt.days

# Calcular la diferencia con la fecha estimada
# Positivo = llegó tarde / Negativo = llegó antes de lo esperado
master_df['delivery_delay_days'] = (master_df['order_delivered_customer_date'] - master_df['order_estimated_delivery_date']).dt.days

# Crear una variable categórica binaria (True/False) que será muy útil para los KPIs en Power BI
master_df['is_late'] = master_df['delivery_delay_days'] > 0

print("Estadísticas de tiempos de entrega (días):")
display(master_df[['delivery_time_days', 'delivery_delay_days']].describe())

Estadísticas de tiempos de entrega (días):


,delivery_time_days,delivery_delay_days
count,110189.000000,110189.000000
mean,12.007342,-12.029041
std,9.451153,10.158194
min,0.000000,-147.000000
25%,6.000000,-17.000000
50%,10.000000,-13.000000
75%,15.000000,-7.000000
max,209.000000,188.000000


# Celda 8: Ingeniería de costos y fletes

In [9]:
# Valor total pagado por el ítem
master_df['total_item_value'] = master_df['price'] + master_df['freight_value']

# Ratio de flete: ¿Qué porcentaje del precio del producto representa el envío?
# Un ratio de 0.5 significa que el envío costó la mitad que el producto.
master_df['freight_ratio'] = master_df['freight_value'] / master_df['price']

# Visualizar el impacto del flete en diferentes categorías
print("\nTop 5 categorías con el flete más caro en proporción al producto:")
top_freight = master_df.groupby('product_category_name_english')['freight_ratio'].mean().sort_values(ascending=False).head()
display(top_freight)


Top 5 categorías con el flete más caro en proporción al producto:


product_category_name_english
home_comfort_2        0.933862
dvds_blu_ray          0.835588
electronics           0.683898
christmas_supplies    0.675458
flowers               0.564979
Name: freight_ratio, dtype: float64

# Celda 9: Preparación del Target (Variable Objetivo)

In [10]:
# Cargar reseñas y limpiar el ID
reviews_df = pd.read_csv(r'E:\Olise TP\CSV\olist_order_reviews_dataset.csv')
reviews_df['order_id'] = reviews_df['order_id'].astype(str).str.strip()

# Nos quedamos con la calificación más reciente por pedido (por si hay duplicados)
reviews_clean = reviews_df.drop_duplicates(subset=['order_id'], keep='last')[['order_id', 'review_score']]

# Unir con nuestro dataframe maestro
df_model = pd.merge(master_df, reviews_clean, on='order_id', how='inner')

# Crear la variable binaria: 1 = Satisfecho (4 o 5 estrellas), 0 = Insatisfecho (1, 2 o 3 estrellas)
df_model['is_satisfied'] = (df_model['review_score'] >= 4).astype(int)

# Convertir variables booleanas a numéricas (1 y 0) para que el modelo las entienda
df_model['is_late'] = df_model['is_late'].astype(int)

print("Distribución de la satisfacción:")
print(df_model['is_satisfied'].value_counts(normalize=True) * 100)

Distribución de la satisfacción:
is_satisfied
1    76.816312
0    23.183688
Name: proportion, dtype: float64


# Celda 10: Selección de Features y División de Datos

In [11]:
from sklearn.model_selection import train_test_split

# Seleccionar las variables predictoras (features) que creamos antes
features = [
    'delivery_time_days', 
    'delivery_delay_days', 
    'freight_ratio', 
    'total_item_value', 
    'is_late'
]

# Filtrar el dataframe para quedarnos solo con lo necesario y eliminar nulos residuales
model_data = df_model[features + ['is_satisfied']].dropna()

X = model_data[features]
y = model_data['is_satisfied']

# Dividir en datos de entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Datos para entrenar: {len(X_train)} filas. Datos para probar: {len(X_test)} filas.")

Datos para entrenar: 87489 filas. Datos para probar: 21873 filas.


# Celda 11: Entrenamiento y Evaluación (Random Forest)

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# Inicializar y entrenar el modelo
# Usamos max_depth=10 para evitar sobreajuste (overfitting) y hacerlo más rápido
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

# Hacer predicciones con los datos de prueba
y_pred = rf_model.predict(X_test)

# Evaluar el rendimiento
print("--- Reporte de Clasificación ---")
print(classification_report(y_test, y_pred))

# Extraer y visualizar la importancia de las variables
importances = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=False)

print("\n--- Importancia de las Variables ---")
display(importances)

--- Reporte de Clasificación ---
              precision    recall  f1-score   support

           0       0.82      0.21      0.33      5071
           1       0.80      0.99      0.89     16802

    accuracy                           0.81     21873
   macro avg       0.81      0.60      0.61     21873
weighted avg       0.81      0.81      0.76     21873


--- Importancia de las Variables ---


delivery_delay_days    0.390316
delivery_time_days     0.218189
is_late                0.200224
freight_ratio          0.098397
total_item_value       0.092874
dtype: float64

# Celda 12: Creación de Tablas de Dimensiones

In [13]:
# 1. Dimensión Clientes (Dim_Customers)
# Cargamos clientes y nos quedamos solo con los que tienen pedidos entregados
customers_df = pd.read_csv(r'E:\Olise TP\CSV\olist_customers_dataset.csv')
customers_df['customer_id'] = customers_df['customer_id'].astype(str).str.strip()

dim_customers = customers_df[['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']]
# Eliminamos duplicados por si acaso
dim_customers = dim_customers.drop_duplicates(subset=['customer_id'])

# 2. Dimensión Productos (Dim_Products)
# Usamos el dataframe 'products_clean' que ya tenía la traducción al inglés y sin nulos
dim_products = products_clean.drop_duplicates(subset=['product_id'])

print(f"Dimensión Clientes: {len(dim_customers)} filas")
print(f"Dimensión Productos: {len(dim_products)} filas")

Dimensión Clientes: 99441 filas
Dimensión Productos: 32951 filas


# Celda 13: Creación de la Tabla de Hechos

In [14]:
# 3. Tabla de Hechos (Fact_Sales)
# Partimos de df_model que ya tiene las variables logísticas y la variable de satisfacción (is_satisfied)
fact_sales = df_model[[
    'order_id', 
    'customer_id', 
    'product_id', 
    'order_purchase_timestamp', 
    'price', 
    'freight_value', 
    'delivery_time_days', 
    'delivery_delay_days',
    'is_late',
    'is_satisfied'
]]

print(f"Tabla de Hechos (Ventas): {len(fact_sales)} filas")

Tabla de Hechos (Ventas): 109370 filas


# Celda 14: Exportación a CSV para Power BI

In [15]:
# Guardamos los archivos limpios en la misma carpeta del proyecto
# index=False evita que pandas exporte la columna del índice numérico
dim_customers.to_csv('BI_dim_customers.csv', index=False)
dim_products.to_csv('BI_dim_products.csv', index=False)
fact_sales.to_csv('BI_fact_sales.csv', index=False)

print("¡Archivos exportados con éxito! Listos para importar en Power BI.")

¡Archivos exportados con éxito! Listos para importar en Power BI.


Hasta este punto, avanzamos a muy buen ritmo y ya dejamos atrás todo el trabajo pesado de procesamiento en Python. Completamos las primeras cinco fases técnicas de tu plan:

1. Exploración y Limpieza (EDA): Analizamos las tablas originales, detectamos dónde faltaba información, filtramos únicamente los pedidos exitosos (delivered), estandarizamos los formatos de fecha y limpiamos los IDs para asegurar que los cruces funcionaran sin fallas.

2. Consolidación de Datos: Construimos nuestro dataset maestro cruzando pedidos, ítems y productos. Resolvimos el problema del idioma traduciendo las categorías al inglés y rescatamos los productos sin categoría etiquetándolos como unknown.

3. Ingeniería de Variables (Features): Creamos los indicadores que realmente le importan al negocio. Calculamos los tiempos exactos de entrega, los días de retraso (delivery_delay_days), un indicador binario de entregas tardías (is_late), y el impacto financiero del costo de envío sobre el producto (freight_ratio).

4. Modelado Predictivo: Integramos las calificaciones de los usuarios y entrenamos un modelo de clasificación (Random Forest) para predecir la satisfacción del cliente, lo que nos permitió identificar qué variables logísticas son las que más destruyen la experiencia de compra.

5. Modelo de Datos Relacional: Para que el motor de BI funcione rápido y sin errores, dividimos el dataset maestro en un Esquema Estrella clásico. Generamos y exportamos tres archivos CSV listos para usar: BI_dim_customers (Dimensión), BI_dim_products (Dimensión) y BI_fact_sales (Tabla de Hechos).